---
title: "Data Collection"
format:
    html: 
        code-fold: false
---


{{< include overview.qmd >}} 

{{< include methods.qmd >}} 

# Code 

## Budget Bytes

The first goal I had was to scrape a recipe from Budget Bytes[@noauthor_budget_2022].  


In [35]:
# Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from fractions import Fraction
import time
import json
from requests.exceptions import HTTPError
import ast
from tqdm import tqdm

In [114]:
%%capture
# Send request to Budget Bytes
url = 'https://www.budgetbytes.com/one-pot-creamy-pesto-chicken-pasta/'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(url, headers = headers)

# Parse the page content using Beautiful Soup
soup = BeautifulSoup(response.content, 'html.parser')

# Examine the entire HTML code
print(soup.prettify())


This request worked and after manually inspecting the "soup" I found that the class for the recipe block is called "wprm-recipe-container". The next goal was to isolate the container with all of the recipe information in it. 

In [115]:
%%capture
# Get the recipe container from the entire soup object
recipe_container = soup.find(class_="wprm-recipe-container")

print(recipe_container.prettify())

Isolating the recipe container worked. The next goal was to parse the HTML to get the recipe title, total cost, cost per serving, user rating, total votes, total servings, prep time, cook time, total time, ingredient measurement values, ingredient measurement units, ingredients, and total steps. The title was relatively easy to extract.

In [116]:
title = recipe_container.find(class_="wprm-recipe-name")

# Strip the accompanying code
title = title.get_text(strip = True)

print(title)

One Pot Creamy Pesto Chicken Pasta


Total cost and cost per serving were a little more difficult and required using regex to split the total cost per recipe and the cost per serving.

In [117]:
cost_text = recipe_container.find("span", class_ = "cost-per").get_text(" ", strip = True)

# Remove all of the letters and the back slash
cost_text = re.sub(r'[a-zA-Z/$]', '', cost_text).strip()

# Split the total cost and cost per serving
costs = cost_text.split(" ")

# Get rid of the empty string
costs = list(filter(None, costs))

# Isolate the total cost
total_cost = costs[0]

# Isolate the cost per serving
cost_per_serving = costs[1]

print("Total cost: ", total_cost)
print("Cost per serving: ", cost_per_serving)

Total cost:  11.77
Cost per serving:  2.94


User rating and total votes were direct pulls.

In [118]:
rating_div = recipe_container.find("div", class_="wprm-recipe-rating")

# Pull the rating info directly from the div
rating_avg = float(rating_div["data-average"])
rating_votes = int(rating_div["data-count"])

print("Average Rating: ", rating_avg)
print("Total Votes: ", rating_votes)

Average Rating:  4.85
Total Votes:  244


Total servings was also a direct pull.

In [119]:
# Combine what we did for to get the recipe name for this one-liner
servings = int(recipe_container.find(class_ = "wprm-recipe-servings").get_text(strip = True))

print("Number of Servings:", servings)

Number of Servings: 4


Prep time, cook time, and total time were also slightly more difficult.  If any of them were greater than an hour Budget Bytes formats it as x hr yy mins. In order to convert that format to only minutes, I utilized regex and a simple time conversion. This method was tested on recipes that only had minutes and recipes that had hours and minutes in the cooking section and it was successful. 

In [120]:
# Find all of the cook time hours and minutes, using similar extraction methods as earlier
# Prep
prep_hours_block = recipe_container.find(class_="wprm-recipe-prep_time-hours")
prep_mins_block = recipe_container.find(class_="wprm-recipe-prep_time-minutes")

prep_hours = prep_hours_block.get_text(strip = True) if prep_hours_block else 0
prep_mins = prep_mins_block.get_text(strip = True) if prep_mins_block else 0

prep_hours = int(re.sub(r'[a-zA-Z]', '', prep_hours).strip() if prep_hours else 0)
prep_mins = int(re.sub(r'[a-zA-Z]', '', prep_mins).strip() if prep_mins else 0)

# Cook
cook_hours_block = recipe_container.find(class_="wprm-recipe-cook_time-hours")
cook_mins_block = recipe_container.find(class_="wprm-recipe-cook_time-minutes")

cook_hours = cook_hours_block.get_text(strip = True) if cook_hours_block else 0
cook_mins = cook_mins_block.get_text(strip = True) if cook_mins_block else 0

cook_hours = int(re.sub(r'[a-zA-Z]', '', cook_hours).strip() if cook_hours else 0)
cook_mins = int(re.sub(r'[a-zA-Z]', '', cook_mins).strip() if cook_mins else 0)

# Total
total_hours_block = recipe_container.find(class_="wprm-recipe-total_time-hours")
total_mins_block = recipe_container.find(class_="wprm-recipe-total_time-minutes")

total_hours = total_hours_block.get_text(strip = True) if total_hours_block else 0
total_mins = total_mins_block.get_text(strip = True) if total_mins_block else 0

total_hours = int(re.sub(r'[a-zA-Z]', '', total_hours).strip() if total_hours else 0)
total_mins = int(re.sub(r'[a-zA-Z]', '', total_mins).strip() if total_mins else 0)

# Convert to minutes
prep_total_minutes = prep_hours * 60 + prep_mins
cook_total_minutes = cook_hours * 60 + cook_mins
total_time_minutes = total_hours * 60 + total_mins

print("Prep Minutes:", prep_total_minutes)
print("Cook Minutes:", cook_total_minutes)
print("Total Minutes:", total_time_minutes)

Prep Minutes: 5
Cook Minutes: 20
Total Minutes: 25


Ingredient measurment values were varied (ie 3, 0.5, 5). This helper function takes in the ingredient measurement string and returns a standardized value.

In [125]:
def parse_amount(amount_str):
    # If there is no amount return none
    if amount_str is None:
        return None
    
    # Strip
    amount_str = amount_str.strip()

    # See if the amount is an integer or decimal
    try:
        return float(amount_str)
    except ValueError:
        pass

    # It appears that if a fraction is greater than one it's listed as a float, but if less than one it's x / y
    try:
        # Clipping the repeating decimals for readability
        return round(float(Fraction(amount_str)), 4)
    except ValueError:
        return None

This loop parses every ingredient in a recipe and extracts the measurement, unit, and name of the ingredient, creating a list of dictionaries.

In [129]:
# Create an empty list for ingredients
ingredients = []

# Find all of the ingredients in the recipe
for ing in recipe_container.find_all("li", class_="wprm-recipe-ingredient"):
    # Get the numerical value for the ingredient
    amount = ing.find(class_="wprm-recipe-ingredient-amount")

    # Get the unit of measurement for the ingredient
    unit = ing.find(class_="wprm-recipe-ingredient-unit")

    # Get the name of the ingredient
    name = ing.find(class_="wprm-recipe-ingredient-name")

    # Append the amount, unit, and ingredient to later map to the USDA nutrition information
    ingredients.append({
        # Use the extraction methods as above 
        # Applying the helper function defined above
        "amount": parse_amount(amount.get_text(strip=True)) if amount else None,
        "unit": unit.get_text(strip=True) if unit else None,
        "ingredient": name.get_text(strip=True) if name else None,
    })

print("Ingredients list: ") 
ingredients


Ingredients list: 


[{'amount': 1.0,
  'unit': 'lb.',
  'ingredient': 'boneless, skinless chicken breast'},
 {'amount': 2.0, 'unit': 'Tbsp', 'ingredient': 'butter'},
 {'amount': 2.0, 'unit': 'cloves', 'ingredient': 'garlic'},
 {'amount': 0.5, 'unit': 'lb.', 'ingredient': 'penne pasta'},
 {'amount': 1.5, 'unit': 'cups', 'ingredient': 'chicken broth'},
 {'amount': 1.0, 'unit': 'cup', 'ingredient': 'milk'},
 {'amount': 3.0, 'unit': 'oz.', 'ingredient': 'cream cheese*'},
 {'amount': 0.3333, 'unit': 'cup', 'ingredient': 'basil pesto'},
 {'amount': 0.25, 'unit': 'cup', 'ingredient': 'grated Parmesan'},
 {'amount': None, 'unit': None, 'ingredient': 'freshly cracked pepper'},
 {'amount': 1.0, 'unit': 'pinch', 'ingredient': 'crushed red pepper'},
 {'amount': 3.0, 'unit': 'cup', 'ingredient': 'fresh spinach'},
 {'amount': 0.25, 'unit': 'cup', 'ingredient': 'sliced sun dried tomatoes'}]

To get the total number of steps, this loop uses the same logic as collecting the ingredients, but only requires the length of the resulting list.

In [137]:
# Empty list for steps
steps = []

# Find all the steps in the recipe
for ins in recipe_container.find_all("li", class_="wprm-recipe-instruction"):
    step = ins.find(class_="wprm-recipe-instruction-text") or ins
    # Get the text only
    steps.append(step.get_text(" ", strip=True))

# Get the total number of steps in the recipe
total_steps = len(steps)

# While I'm here I'll get the total length of the instructions to help with calculating a complexity score
step_len = len("".join(steps))

print("Total Steps: ", total_steps)
print("Length of Steps: ", step_len)

Total Steps:  6
Length of Steps:  1422


I was able to successfully parse a single recipe from Budget Bytes. The next step was to parse multiple recipes to build a dataset. Budget Bytes has a "Surprise Me" button that will load a random recipe and I wanted to utilize that to get the samples.

In [ ]:
# Url that redirects user to a random recipe
random_url = "https://www.budgetbytes.com/random/"

response = requests.get(random_url, headers = headers)

# Chec to see if an error was raised
response.raise_for_status()

# Return the url 
response.url

'https://www.budgetbytes.com/twice-baked-potato-casserole/'

I created a function that get's a random recipe from Budget Bytes and extracts the data using the methods outlined abovd.

In [152]:
def sample_and_parse_recipes(n = 10, sleep_sec = 2, out_path = "../../data/raw-data/budgetbytes_recipes.csv"):
    # Set the headers again so we don't receive a security error
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    }

    # Define urls as a set for now so we don't get repeats and recipes as an empty list
    urls_seen = set()
    recipes = []

    # I don't think we would get duplicates but this loop accounts for them
    while len(recipes) < n:
        # Url that redirects user to a random recipe
        random_url = "https://www.budgetbytes.com/random/"

        response = requests.get(random_url, headers = headers)

        # Check to see if an error was raised
        response.raise_for_status()

        # Track duplicates
        final_url = response.url
        if final_url in urls_seen:
            time.sleep(sleep_sec)
            continue
        urls_seen.add(final_url)

        # Parse the page content using Beautiful Soup
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Get the recipe container from the entire soup object
        recipe_container = soup.find(class_="wprm-recipe-container")
        # If the recipe container doesn't exist for whatever reason skip it
        if recipe_container is None:
            time.sleep(sleep_sec)
            continue

        title = recipe_container.find(class_="wprm-recipe-name")

        # Strip the accompanying code
        title = title.get_text(strip = True)

        cost_text = recipe_container.find("span", class_ = "cost-per")

        if cost_text:
            cost_text = cost_text.get_text(" ", strip = True)
            # Remove all of the letters and the back slash
            cost_text = re.sub(r'[a-zA-Z/$]', '', cost_text).strip()
            # Get rid of the empty string
            costs = list(filter(None, cost_text.split(" ")))
        else:
            costs = []

        # Isolate the total cost
        total_cost = costs[0] if len(costs) >= 1 else None

        # Isolate the cost per serving
        cost_per_serving = costs[1] if len(costs) >= 2 else None

        rating_div = recipe_container.find("div", class_="wprm-recipe-rating")

        # Pull the rating info directly from the div
        rating_avg = float(rating_div["data-average"])
        rating_votes = int(rating_div["data-count"])

        # Combine what we did for to get the recipe name for this one-liner
        servings = int(recipe_container.find(class_ = "wprm-recipe-servings").get_text(strip = True))

        # Find all of the cook time hours and minutes, using similar extraction methods as earlier
        # Prep
        prep_hours_block = recipe_container.find(class_="wprm-recipe-prep_time-hours")
        prep_mins_block = recipe_container.find(class_="wprm-recipe-prep_time-minutes")

        prep_hours = prep_hours_block.get_text(strip = True) if prep_hours_block else 0
        prep_mins = prep_mins_block.get_text(strip = True) if prep_mins_block else 0

        prep_hours = int(re.sub(r'[a-zA-Z]', '', prep_hours).strip() if prep_hours else 0)
        prep_mins = int(re.sub(r'[a-zA-Z]', '', prep_mins).strip() if prep_mins else 0)

        # Cook
        cook_hours_block = recipe_container.find(class_="wprm-recipe-cook_time-hours")
        cook_mins_block = recipe_container.find(class_="wprm-recipe-cook_time-minutes")

        cook_hours = cook_hours_block.get_text(strip = True) if cook_hours_block else 0
        cook_mins = cook_mins_block.get_text(strip = True) if cook_mins_block else 0

        cook_hours = int(re.sub(r'[a-zA-Z]', '', cook_hours).strip() if cook_hours else 0)
        cook_mins = int(re.sub(r'[a-zA-Z]', '', cook_mins).strip() if cook_mins else 0)

        # Total
        total_hours_block = recipe_container.find(class_="wprm-recipe-total_time-hours")
        total_mins_block = recipe_container.find(class_="wprm-recipe-total_time-minutes")

        total_hours = total_hours_block.get_text(strip = True) if total_hours_block else 0
        total_mins = total_mins_block.get_text(strip = True) if total_mins_block else 0

        total_hours = int(re.sub(r'[a-zA-Z]', '', total_hours).strip() if total_hours else 0)
        total_mins = int(re.sub(r'[a-zA-Z]', '', total_mins).strip() if total_mins else 0)

        # Convert to minutes
        prep_total_minutes = prep_hours * 60 + prep_mins
        cook_total_minutes = cook_hours * 60 + cook_mins
        total_time_minutes = total_hours * 60 + total_mins

        def parse_amount(amount_str):
            # If there is no amount return none
            if amount_str is None:
                return None
            
            # Strip
            amount_str = amount_str.strip()

            # See if the amount is an integer or decimal
            try:
                return float(amount_str)
            except ValueError:
                pass

            # It appears that if a fraction is greater than one it's listed as a float, but if less than one it's x / y
            try:
                # Clipping the repeating decimals for readability
                return round(float(Fraction(amount_str)), 4)
            except ValueError:
                return None
        
        # Create an empty list for ingredients
        ingredients = []

        # Find all of the ingredients in the recipe
        for ing in recipe_container.find_all("li", class_="wprm-recipe-ingredient"):
            # Get the numerical value for the ingredient
            amount = ing.find(class_="wprm-recipe-ingredient-amount")

            # Get the unit of measurement for the ingredient
            unit = ing.find(class_="wprm-recipe-ingredient-unit")

            # Get the name of the ingredient
            name = ing.find(class_="wprm-recipe-ingredient-name")

            # Append the amount, unit, and ingredient to later map to the USDA nutrition information
            ingredients.append({
                # Use the extraction methods as above 
                # Applying the helper function defined above
                "amount": parse_amount(amount.get_text(strip=True)) if amount else None,
                "unit": unit.get_text(strip=True) if unit else None,
                "ingredient": name.get_text(strip=True) if name else None,
            })
        
        # Empty list for steps
        steps = []

        # Find all the steps in the recipe
        for ins in recipe_container.find_all("li", class_="wprm-recipe-instruction"):
            step = ins.find(class_="wprm-recipe-instruction-text") or ins
            # Get the text only
            steps.append(step.get_text(" ", strip=True))

        # Get the total number of steps in the recipe
        total_steps = len(steps)

        # While I'm here I'll get the total length of the instructions to help with calculating a complexity score
        step_len = len("".join(steps))

        # Sleep
        time.sleep(sleep_sec)

        recipe = {
        "title": title,
        "total_cost": total_cost,
        "cost_per_serving": cost_per_serving,
        "rating_avg": rating_avg,
        "rating_vote": rating_votes,
        "servings": servings,
        "prep_time_min": prep_total_minutes,
        "cook_time_min": cook_total_minutes,
        "total_time_min": total_time_minutes,
        "ingredients": ingredients,
        "num_steps": total_steps,
        "step_len": step_len,
        }

        recipes.append(recipe)
    
    # Make a dataframe from all of the recipes
    df = pd.DataFrame(recipes)

    # Save the dataframe
    df.to_csv(out_path, index = False)

    # Return the dataframe
    return df

I tested the function with a small number of recipes.

In [147]:
recipes = sample_and_parse_recipes(5, 2)

I then tested it with a medium number.

In [154]:
recipes = sample_and_parse_recipes(50, 2)

Budget Bytes claims to have a total of 1200+ recipes, so I settled on trying to scrape 800. One shortfall of using my random recipe collection method is that once it has collected the majority of the recipes, the function will be making exponential repeat requests as the number of requested recipes gets closer to 1200.

In [155]:
recipes = sample_and_parse_recipes(800, 2)

Before utilizing the USDA FoodData API I wanted to do some intermediary normalization on the ingredients to make the mapping of Budget Bytes ingredients easier. I first checked to see how many unique ingredients are in the Budget Bytes dataset.

In [9]:
recipes = pd.read_csv("../../data/raw-data/budgetbytes_recipes.csv")
recipes["ingredients"] = recipes["ingredients"].apply(ast.literal_eval)

In [10]:
all_ingredients = []

# The ingredients are stored as a list of dictionaries so let's loop through them
for ing_list in recipes["ingredients"]:
    for ing in list(ing_list):
        # Make sure an ingredient exists
        if ing["ingredient"]:
            all_ingredients.append(ing["ingredient"])

# set to get unique ingredients
unique_ingredients = list(set(all_ingredients))

len(unique_ingredients)

2145

There were 2145 total ingredients in all 800 recipes. I then wanted to see how many of them were unique.

In [11]:
unique_ingredients[0:10]

['cheddar cheese',
 'carrots (about 4)',
 'freshly cracked pepper to taste',
 'semi-sweet chocolate, chopped*',
 'red wine**',
 'butter, room temperature',
 'grated ginger',
 'old-fashioned rolled oats',
 'sliced peaches in their juices',
 'rice cereal*']

I then created a function to lightly cleaned the ingredients list to remove common preparation words, trailing phrases, size, and quality [@gpt5.1_cooking_phrases]. I also applied common text cleaning methods like trimming whitespace and stripping special characters. 

In [12]:
# Define a helper function to clean the ingredients
def light_clean_ingredient(ingredient):
    # Define some of the prep words we might see in the dataset
    prep_words = ["chopped", "diced", "pressed", "minced", "sliced", "shredded",
        "peeled", "trimmed", "seeded", "softened", "melted", "divided",
        "uncooked", "cooked"]
    
    # Define some commonly seen phrases in the ingredients
    trailing_phrases = [
        "for garnish", "for serving", "to taste", "or to taste",
        "room temperature", "room-temp", "room temp",
        "divided", "optional", "about", "roughly"
    ]

    # Commonly seen words describing size
    size_words = [
        "small", "medium", "large",
        "baby", "jumbo", "extra-large", "extra large",
        "x-large", "xl", "mini"
    ]

    # Commonly seen words describing quality
    quality_words = [
        "fresh", "frozen", "canned", "dried",
        "unsalted", "salted", "low-sodium", "low sodium",
        "reduced-sodium", "reduced sodium",
        "whole", "lean"
    ]
    
    # lowercase and trim whitespace
    ingredient = ingredient.lower().strip()

    # remove the parentheses and contents
    ingredient = re.sub(r"\(.*?\)", "", ingredient).strip()

    # remove dollar amounts and *
    ingredient = re.sub(r"\$[\d\.\+]+", "", ingredient)
    ingredient = ingredient.replace("*", "").strip()

    # remove commas
    ingredient = ingredient.replace(",", " ").strip()

    # Drop the commonly seen phrases
    for phrase in trailing_phrases:
        ingredient = ingredient.replace(phrase, "")

    # remove the prep words
    no_preps = [word for word in ingredient.split() if word not in prep_words]
    ingredient = " ".join(no_preps).strip()

    # remove the size words
    no_sizes = [word for word in ingredient.split() if word not in size_words]
    ingredient = " ".join(no_sizes).strip()

    # remove the quality words
    no_quality = [word for word in ingredient.split() if word not in quality_words]
    ingredient = " ".join(no_quality).strip()

    # replace multiple spaces
    ingredient = re.sub(r"\s+", " ", ingredient)

    return ingredient

The next step was to apply that cleaning function to the full list of ingredients.

In [13]:
light_cleaned_ingredients = []
for ing in unique_ingredients:
    light_cleaned_ingredients.append(light_clean_ingredient(ing))

light_cleaned_ingredients = list(set(light_cleaned_ingredients))

len(light_cleaned_ingredients)

1292

The light cleaning was able to nearly halved all of the ingredients. It was applied to the recipe dataframe.

In [14]:
for ing_list in recipes["ingredients"]:
    for ing in ing_list:
        raw = ing.get("ingredient")
        if raw:
            ing["ingredient_clean"] = light_clean_ingredient(raw)
        else:
            ing["ingredient_clean"] = None

A list of unique ingredients was created to use as queries for the USDA Food Data API.

In [15]:
# Create lists to store the raw and clean ingredients
all_raw = []
all_clean = []

# Similar to what I did above, just adding the mapping to both lists
for ing_list in recipes["ingredients"]:
    for ing in ing_list:
        raw = ing.get("ingredient")
        if raw:
            cleaned = light_clean_ingredient(raw)
            all_raw.append(raw)
            all_clean.append(cleaned)

ingredient_map = pd.DataFrame({"ingredient_raw": all_raw, "ingredient_clean": all_clean}).drop_duplicates()

ingredient_map["ingredient_clean"].nunique()

1292

In [16]:
unique_clean = ingredient_map["ingredient_clean"].dropna().unique()

The next step was to test out the USDA FoodDATA API, which was tested on chicken. The USDA FoodDATA API has several databases available [@noauthor_api_nodate].

* SR Legacy is the older standard that is no longer updated and calculated nutrition per 100 g/ml
* FNDDS is a link to NHANES dietary intake data
* Foundation contains basic foods
* Branded offers detailed information on brand-specific foods

In [17]:
# Load api key
with open('/home/c18ty/.api-keys.json') as f:
    keys = json.load(f)
API_KEY = keys['usda']

url = "https://api.nal.usda.gov/fdc/v1/foods/search"

params = {
    "api_key": API_KEY,
    "query": "chicken",
    # Get the top match
    "pageSize": 1, 
    "dataType": ["SR Legacy", "Survey (FNDDS)", "Foundation", "Branded"]
}

response = requests.get(url, params=params)
response.raise_for_status
data = response.json()
food = data.get("foods")
print(food)

[{'fdcId': 2038064, 'description': 'CHICKEN', 'dataType': 'Branded', 'gtinUpc': '796853100065', 'publishedDate': '2021-10-28', 'brandOwner': 'Keystone Brand Meats Inc', 'brandName': 'KEYSTONE', 'ingredients': 'CHICKEN, SEA SALT.', 'marketCountry': 'United States', 'foodCategory': 'Canned Meat', 'modifiedDate': '2019-05-15', 'dataSource': 'LI', 'packageWeight': '14.5 oz/411 g', 'servingSizeUnit': 'g', 'servingSize': 56.0, 'householdServingFullText': '2 ONZ', 'tradeChannels': ['NO_TRADE_CHANNEL'], 'allHighlightFields': '<b>Ingredients</b>: <em>CHICKEN</em>, SEA SALT.', 'score': 959.9012, 'microbes': [], 'foodNutrients': [{'nutrientId': 1003, 'nutrientName': 'Protein', 'nutrientNumber': '203', 'unitName': 'G', 'derivationCode': 'LCCS', 'derivationDescription': 'Calculated from value per serving size measure', 'derivationId': 70, 'value': 21.4, 'foodNutrientSourceId': 9, 'foodNutrientSourceCode': '12', 'foodNutrientSourceDescription': "Manufacturer's analytical; partial documentation", 'ra

The search returned the top ingredient searching all of the available ingredients. This was able to be parsed for the nutrients.

In [18]:
food1 = food[0]
nutrients = {nut["nutrientName"]: nut["value"] for nut in food1.get("foodNutrients", [])}
nutrients

{'Protein': 21.4,
 'Total lipid (fat)': 1.79,
 'Carbohydrate, by difference': 0.0,
 'Energy': 107,
 'Total Sugars': 0.0,
 'Fiber, total dietary': 0.0,
 'Calcium, Ca': 0.0,
 'Iron, Fe': 0.0,
 'Sodium, Na': 179,
 'Vitamin A, IU': 0.0,
 'Vitamin C, total ascorbic acid': 0.0,
 'Cholesterol': 45.0,
 'Fatty acids, total trans': 0.0,
 'Fatty acids, total saturated': 0.89}

 then created a function that extracted all of the nutrient information to be mapped to the Budget Bytes ingredients.

In [34]:
def parse_usda_food(food):
    
    nutrients = {}
    for nut in food.get("foodNutrients", []):
        name = nut.get("nutrientName")
        value = nut.get("value")
        if name is not None and value is not None:
            nutrients[name] = value

    # Nutrition information per serving
    calories = nutrients.get("Energy", None)                      
    protein = nutrients.get("Protein", None)                     
    fat = nutrients.get("Total lipid (fat)", None)     
    carbs = nutrients.get("Carbohydrate, by difference", None)

    # Serving size
    serving_size = food.get("servingSize")
    serving_size_unit = food.get("servingSizeUnit")

    # Number of sub ingredients
    ingredients_str = food.get("ingredients", "") or ""
    sub_ingredients = [sub.strip() for sub in ingredients_str.split(",") if sub.strip()]
    ingredient_count = len(sub_ingredients)

    return {
        "fdc_id": food.get("fdcId"),
        "description": food.get("description"),
        "serving_size": serving_size,
        "serving_size_unit": serving_size_unit,
        "ingredients_str": ingredients_str,
        "ingredient_count": ingredient_count,
        "calories": calories,
        "protein": protein,
        "fat": fat,
        "carbs": carbs,
    }

parse_usda_food(food[0])

{'fdc_id': 171024,
 'description': 'Oil, cottonseed, salad or cooking',
 'serving_size': None,
 'serving_size_unit': None,
 'ingredients_str': '',
 'ingredient_count': 0,
 'calories': 3700.0,
 'protein': 0.0,
 'fat': 100,
 'carbs': 0.0}

After some experimenting, I found that a single search alone didn't provide the best results. I decided to pull the SR Legacy data and the Branded data for comparison, and defined functions for each. I was hit with rate limits, which I found that the USDA FoodData API limits requests to 1000 per hour, so I had to implement a backoff for these requests.

In [33]:
def usda_search_sr(query, api = API_KEY, max_retries = 5, base_sleep = 0.5):
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"

    params = {
        "api_key": API_KEY,
        "query": query,
        # Get the top match
        "pageSize": 1, 
        "dataType": ["SR Legacy"]
    }

    response = None

    for attempt in range(max_retries):
        response = requests.get(url, params=params)

        # Handling for rate limit
        if response.status_code == 429:
            # USDA sent me a "retry after"
            retry_after = response.headers.get("Retry-After")
            if retry_after is not None:
                sleep_sec = float(retry_after)
            else:
                # back-off
                sleep_sec = base_sleep * (2 ** attempt)
            print(f"Rate limited on '{query}'. Sleeping {sleep_sec:.1f}s...")
            time.sleep(sleep_sec)
            continue

        break

    if response is None:
        return None

    response.raise_for_status()
    data = response.json()
    return data.get("foods")

In [32]:
def usda_search_branded(query, api = API_KEY, max_retries = 5, base_sleep = 0.5):
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"

    params = {
        "api_key": API_KEY,
        "query": query,
        # Get the top match
        "pageSize": 1, 
        "dataType": ["Branded"]
    }

    response = None

    for attempt in range(max_retries):
        response = requests.get(url, params=params)

        # Handling for rate limit
        if response.status_code == 429:
            # USDA sent me a "retry after"
            retry_after = response.headers.get("Retry-After")
            if retry_after is not None:
                sleep_sec = float(retry_after)
            else:
                # back-off
                sleep_sec = base_sleep * (2 ** attempt)
            print(f"Rate limited on '{query}'. Sleeping {sleep_sec:.1f}s...")
            time.sleep(sleep_sec)
            continue

        break

    if response is None:
        return None
    
    data = response.json()
    return data.get("foods")

Nutrition information was collected on the SR Legacy Data. Errors were received on queries with fractions in them, which was a limitation of my query cleaning.

In [36]:
# Empty list for our results
rows = []

# Itterate through every unique ingredient in our list
for name in tqdm(unique_clean, desc = "SR Legacy"):
    # Search the ingredient, handle error so entire loop doesn't break
    try:
        food = usda_search_sr(name)
    except HTTPError as e:
        print(f"USDA error for {name}: {e}. Skipping.")
        food = None

    # Get the information we are interested in, nothing if no responses
    if food and len(food) > 0:
        parsed = parse_usda_food(food[0])
    else:
        parsed = {
        "fdc_id": None,
        "description": None,
        "serving_size": None,
        "serving_size_unit": None,
        "ingredients_str": None,
        "ingredient_count": None,
        "calories": None,
        "protein": None,
        "fat": None,
        "carbs": None,
    }
    
    # Add the clean ingredient name for mapping later
    parsed["ingredient_clean"] = name
    rows.append(parsed)

# Create a dataframe with all of the results
usda_nutrition_sr = pd.DataFrame(rows)

usda_nutrition_sr.to_csv("../../data/raw-data/usda_nutrition_sr.csv", index = False)

SR Legacy:   0%|          | 0/1292 [00:00<?, ?it/s]

SR Legacy:  72%|███████▏  | 932/1292 [08:56<03:20,  1.79it/s]

USDA error for onion in 1/4 inch half moons: 500 Server Error: Internal Server Error for url: https://api.nal.usda.gov/fdc/v1/foods/search?api_key=E2GPPmsbNWHgJrOMRXq8eYe5ha7ONl30WSqDQvth&query=onion+in+1%2F4+inch+half+moons&pageSize=1&dataType=SR+Legacy. Skipping.


SR Legacy:  72%|███████▏  | 933/1292 [08:56<03:11,  1.88it/s]

USDA error for bell pepper into 1/4-inch strips: 500 Server Error: Internal Server Error for url: https://api.nal.usda.gov/fdc/v1/foods/search?api_key=E2GPPmsbNWHgJrOMRXq8eYe5ha7ONl30WSqDQvth&query=bell+pepper+into+1%2F4-inch+strips&pageSize=1&dataType=SR+Legacy. Skipping.


...

SR Legacy:  87%|████████▋ | 1119/1292 [50:38<01:39,  1.75it/s]

USDA error for garlic or 1/4 tsp garlic powder: 500 Server Error: Internal Server Error for url: https://api.nal.usda.gov/fdc/v1/foods/search?api_key=E2GPPmsbNWHgJrOMRXq8eYe5ha7ONl30WSqDQvth&query=garlic+or+1%2F4+tsp+garlic+powder&pageSize=1&dataType=SR+Legacy. Skipping.


SR Legacy:  96%|█████████▌| 1236/1292 [51:52<00:32,  1.71it/s]

USDA error for carrots 1/4-inch rounds: 500 Server Error: Internal Server Error for url: https://api.nal.usda.gov/fdc/v1/foods/search?api_key=E2GPPmsbNWHgJrOMRXq8eYe5ha7ONl30WSqDQvth&query=carrots+1%2F4-inch+rounds&pageSize=1&dataType=SR+Legacy. Skipping.


SR Legacy: 100%|██████████| 1292/1292 [52:25<00:00,  2.43s/it]


Nutrition information was also collected for the Branded data.

In [ ]:
# Empty list for our results
rows = []

# Itterate through every unique ingredient in our list
for name in tqdm(unique_clean, desc = "Branded"):
    # Search the ingredient, handle error so entire loop doesn't break
    try:
        food = usda_search_branded(name)
    except HTTPError as e:
        print(f"USDA error for {name}: {e}. Skipping.")
        food = None

    # Get the information we are interested in, nothing if no responses
    if food and len(food) > 0:
        parsed = parse_usda_food(food[0])
    else:
        parsed = {
        "fdc_id": None,
        "description": None,
        "serving_size": None,
        "serving_size_unit": None,
        "ingredients_str": None,
        "ingredient_count": None,
        "calories": None,
        "protein": None,
        "fat": None,
        "carbs": None,
    }
    
    # Add the clean ingredient name for mapping later
    parsed["ingredient_clean"] = name
    rows.append(parsed)

# Create a dataframe with all of the results
usda_nutrition_branded = pd.DataFrame(rows)

usda_nutrition_branded.to_csv("../../data/raw-data/usda_nutrition_branded.csv", index = False)

{{< include closing.qmd >}} 